## TODO
- В get_cookie_recency добавить recency_contact (как заполнять nan? np.inf?)
- Доп фичи:
    - df_cookie_category_features !
    - cookie_surface_count
    - cookie_top_surface
    - distance cookie node projections (может еще с category projection?)

- Добавить в pool(cat_features=[...])
- Добавить в расстояния - L1, L2
- Понизить тип float64 для als_rank и lightfm_rank

К Толику:
- Как эффективно считать расстояния между векторами (map_elements, map_batches)?
- 

In [1]:
VERSION = '50'
DEVICE = 'GPU'

PARAMS = dict(
    als_iterations = 15,
    als_factors = 700,
    als_regularization = 0.45,
    als_alpha = 4,
    als_w_contact = 41,
    als_N_cand = 500,
    
    lightfm_factors = 150,
    lightfm_loss = 'warp',
    lightfm_lr = 0.05,
    lightfm_epochs = 20,
    lightfm_N_cand = 300,
    
    pop_N_cand = 50,
    proj_N_cand = 50,

    cb_learning_rate = 0.3,
    cb_iterations = 1000,
    cb_depth = 8,
    cb_l2_leaf_reg = 1.0,
    cb_loss_function = 'YetiRank',
    
    do_test_pred = True,
)

DATA_DIR  = '/kaggle/input/avito-cup-2025-recsys'
CACHE_DIR = '/kaggle/input/aaa-recsys-ods-dataset'

FILES = dict(
    VAL = dict(
        cands_als         = f'{CACHE_DIR}/cands_als_VAL_v46_n500.parquet',
        cands_lightfm     = f'{CACHE_DIR}/cands_lightfm_VAL_v43_n300.parquet',
        cands_pop         = f'{CACHE_DIR}/cands_pop_VAL_v41_n50.parquet',
        cands_proj        = f'{CACHE_DIR}/cands_proj_VAL_v41_n50.parquet',
        projection_cookie = f'{CACHE_DIR}/projection_cookie_VAL_v42.parquet',
    ),
    
    TEST = dict(
        cands_als         = f'{CACHE_DIR}/cands_als_TEST_v46_n500.parquet',
        cands_lightfm     = f'{CACHE_DIR}/cands_lightfm_TEST_v43_n300.parquet',
        cands_pop         = f'{CACHE_DIR}/cands_pop_TEST_v41_n50.parquet',
        cands_proj        = f'{CACHE_DIR}/cands_proj_TEST_v41_n50.parquet',
        projection_cookie = f'{CACHE_DIR}/projection_cookie_TEST_v42.parquet',
    )
)

PROJECTION_NODE_PATH   = f'{CACHE_DIR}/projection_node.parquet'

In [2]:
!pip install implicit mlflow faiss-cpu lightfm rectools polars==1.25.2 >> _

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 3.6.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2025.3.2 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
bigframes 1.42.0 requires rich<14,>=12.4.4, but you have rich 14.0.0 which is incompatible.
cudf-polars-cu12 25.2.2 requires polars<1.22,>=1.20, but you have polars 1.25.2 which is incompatible.
plotnine 0.14.5 requires matplotlib>=3.8.0, but you have matplotlib 3.7.2 which is incompatible.
pandas-gbq 0.28.0 requires google-api-core<3.0.0dev,>=2.10.2, but you have google-api-core 1.34.1 which is incompatible.
mlxtend 0.23.4 requires scikit-learn>=1.3.1, but you have scikit-learn 1.2.2 which is incompatible.


In [3]:
import os
import gc
import pickle
import random
from datetime import timedelta, datetime
from tqdm import tqdm

import numpy as np
from numpy.linalg import norm
import pandas as pd
import polars as pl
import implicit

from lightfm import LightFM
from lightfm.data import Dataset

from scipy.sparse import csr_matrix
from sklearn.model_selection import GroupShuffleSplit
from catboost import Pool, CatBoostRanker

from rectools.models import SASRecModel
from rectools.dataset import Dataset as rectools_dataset

import faiss
import mlflow

import warnings
warnings.simplefilter('ignore')


SEED = 27
np.random.seed(SEED)
random.seed(SEED)

mlflow.set_tracking_uri("http://51.250.35.156:5000/")
mlflow.set_experiment("homework_astrofimuk")

<Experiment: artifact_location='mlflow-artifacts:/', creation_time=1745951071722, experiment_id='27', last_update_time=1745951071722, lifecycle_stage='active', name='homework_astrofimuk', tags={}>

In [4]:
def get_data() -> tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame, pl.DataFrame, pl.DataFrame]:
    """
    Load and preprocess the core datasets.

    Returns:
        df_test (pl.DataFrame): test users to predict nodes
        df_clickstream (pl.DataFrame): clickstream joined with events (is_contact)
        df_events (pl.DataFrame): event metadata
        df_cat (pl.DataFrame): categorical features to items
        df_text (pl.DataFrame): text feature projections to items
    """
    df_test        = pl.read_parquet(f'{DATA_DIR}/test_users.pq').select(pl.all().shrink_dtype())
    df_clickstream = pl.read_parquet(f'{DATA_DIR}/clickstream.pq').select(pl.all().shrink_dtype())
    df_events      = pl.read_parquet(f'{DATA_DIR}/events.pq').select(pl.all().shrink_dtype())
    # df_text        = pl.read_parquet(f'{DATA_DIR}/text_features.pq').select(pl.all().shrink_dtype())
    df_cat         = (
        pl.read_parquet(f'{DATA_DIR}/cat_features.pq')
        .filter(~pl.col('category').is_null()) # 1 row with missing data (item = 28737)
        .select(pl.all().shrink_dtype())
    )

    df_clickstream = (df_clickstream.join(df_events, on='event', how='left'))
    return df_test, df_clickstream, df_cat  #, df_text


def split_train_val(
    df_clickstream: pl.DataFrame, 
    days: int = 14
) -> tuple[pl.DataFrame, pl.DataFrame]:
    """
    Split clickstream history into training and validation datasets.

    Args:
        df_clickstream: full click history with 'event_date'
        days: how many days at end to use as validation

    Returns:
        df_train: interactions up to cutoff
        df_val: new unique (cookie, node) pairs with contact in the holdout period 
    """
    threshold = df_clickstream['event_date'].max() - timedelta(days=days)

    df_train = df_clickstream.filter(df_clickstream['event_date'] <= threshold)
    df_eval = (
        df_clickstream
        .filter(df_clickstream['event_date'] > threshold)
        .select(['cookie', 'node', 'event', 'is_contact'])
        .join(df_train, on=['cookie', 'node'], how='anti')
        .filter(pl.col('is_contact') == 1)
        .filter(pl.col('cookie').is_in(df_train['cookie'].unique()))
        .filter(pl.col('node').is_in(df_train['node'].unique()))
        .unique(['cookie', 'node'])
    )
    return df_train, df_eval


def get_labels(
    df_val: pl.DataFrame, 
    df_cands: pl.DataFrame
) -> pl.DataFrame:
    """
    Join candidate pairs (cookie, node)  with true labels for validation.

    Args:
        df_val: true (cookie, node) contact pairs
        df_cands: candidate (cookie, node, ...) rows

    Returns:
        df_cands labeled with 0/1 in 'label' column
    """
    df_positive_val = (
        df_val
        .select(['cookie','node'])
        .with_columns(pl.lit(1).alias('label'))
    )
    
    df_cands = (
        df_cands
        .join(df_positive_val, on=['cookie','node'], how='left')
        .with_columns(pl.col('label').fill_null(0).shrink_dtype().alias('label'))
    )
    return df_cands
    

def split_cands_train_val(df_cands: pl.DataFrame) -> tuple[pl.DataFrame, pl.DataFrame]:
    """
    Split candidates for ranker training (80%) and validation (20%) by cookie group.

    Args:
        df_cands: full candidate DataFrame with 'cookie' and 'label'

    Returns:
        df_train: training subset for ranker
        df_val: validation subset for ranker
    """
    df_cands = df_cands.to_pandas()
    
    gss = GroupShuffleSplit(test_size=0.2, random_state=SEED)
    train_idx, val_idx = next(gss.split(
        X=df_cands,
        y=df_cands['label'],
        groups=df_cands['cookie']
    ))
    df_cands_train = pl.from_pandas(df_cands.iloc[train_idx])
    df_cands_val   = pl.from_pandas(df_cands.iloc[val_idx])
    return df_cands_train, df_cands_val


def recall_candidates(df_val: pl.DataFrame, df_cands: pl.DataFrame) -> float:
    """
    Candidate-stage recall: fraction of true contacts covered by candidates in df_cands.

    Args:
        df_val: true (cookie, node) contact pairs
        df_cands: candidate (cookie, node) pairs

    Returns:
        Average recall over cookies
    """
    df_true = df_val.select(['cookie','node']).unique()
    df_hit = df_true.join(
        df_cands.select(['cookie','node']),
        on=['cookie','node'],
        how='inner'
    )
    per_user = (
        df_true
        .group_by('cookie')
        .agg(pl.count().alias('true_cnt'))
        .join(
            df_hit.group_by('cookie').agg(pl.count().alias('hit_cnt')),
            on='cookie', how='left'
        )
        .with_columns(
            (pl.col('hit_cnt').fill_null(0) / pl.col('true_cnt'))
            .alias('recall_cand')
        )
    )
    return per_user['recall_cand'].mean()


def recall_at(
    df_true: pl.DataFrame,
    df_pred: pl.DataFrame,
    k: int = 40,
) -> float:
    """
    Compute Recall@k: fraction of true nodes appearing in top-k predictions per cookie.

    Args:
        df_true: true (cookie, node) pairs
        df_pred: scored predictions with 'cookie', 'node', 'score'
        k: cutoff

    Returns:
        scalar recall@k averaged over cookies
    """
    # sanity checks
    assert df_pred.group_by(['cookie']).agg(pl.col('node').count())['node'].max() < 41 , 'send more then 40 nodes per cookie'
    assert 'node' in df_pred.columns, 'node columns does not exist'
    assert 'cookie' in df_pred.columns, 'cookie columns does not exist'
    assert df_pred.with_columns(v = 1).group_by(['cookie','node']).agg(pl.col('v').count())['v'].max() == 1 , 'more then 1 cookie-node pair'
    assert df_pred['cookie'].dtype == pl.Int64, 'cookie must be int64'
    assert df_pred['node'].dtype == pl.Int64, 'node must be int64'
    
    return (df_true[['node', 'cookie']]
            .join(
                df_pred.group_by('cookie').head(k).with_columns(value=1)[['node', 'cookie', 'value']],
                how='left',
                on=['cookie', 'node']
            )
            .select([pl.col('value').fill_null(0), 'cookie'])
            .group_by('cookie')
            .agg([
                pl.col('value').sum() / pl.col('value').count()
            ])['value']
            .mean()
           )

In [5]:
class ALSRecommender:
    """
    ALS-based collaborative filtering recommender.
    """
    
    def __init__(self, params: dict[str, float]):
        self.params = {key: value for key, value in params.items() if key.startswith('als_')}
        
        self.iterations     = self.params['als_iterations']
        self.factors        = self.params['als_factors']
        self.regularization = self.params['als_regularization']
        self.alpha          = self.params['als_alpha']
        self.N_cand         = self.params['als_N_cand']
        self.w_contact      = self.params['als_w_contact']
        
        self.user_to_index = {}
        self.node_to_index = {}
        self.index_to_node = {}
        self.matrix = None
        self.model = None

    def fit(self, df_train: pl.DataFrame):
        users = df_train['cookie'].to_list()
        nodes = df_train['node'].to_list()
        # build mappings
        self.user_to_index = {u: i for i, u in enumerate(sorted(set(users)))}
        self.node_to_index = {n: j for j, n in enumerate(sorted(set(nodes)))}
        self.index_to_node = {j: n for n, j in self.node_to_index.items()}
        # build sparse matrix
        rows = [self.user_to_index[u] for u in users]
        cols = [self.node_to_index[n] for n in nodes]
        data = df_train['is_contact'].map_elements(
            lambda el: 
            self.w_contact if el == 1 
            else 1
        ).to_numpy()
        
        self.matrix = csr_matrix(
            (data, (rows, cols)), 
            shape=(len(self.user_to_index), len(self.node_to_index))
        )
        
        self.model = implicit.als.AlternatingLeastSquares(
            iterations=self.iterations,
            factors=self.factors,
            regularization=self.regularization,
            alpha=self.alpha,
            num_threads=1,
            random_state=SEED
        )
        self.model.fit(self.matrix)
        return self

    def recommend(self, df_users: pl.DataFrame ) -> pl.DataFrame:
        """
        Generate top-als_N_cand recommendations per cookie.
        """
        user_list = df_users['cookie'].unique().to_list()
        idxs = [self.user_to_index[c] for c in user_list]
        recs, scores = self.model.recommend(
            userid=idxs,
            user_items=self.matrix[idxs],
            N=self.N_cand,
            filter_already_liked_items=True
        )
        
        df_pred = pl.DataFrame({
            'cookie': np.repeat(user_list, self.N_cand),
            'node':  np.concatenate([[self.index_to_node[j] for j in r] for r in recs]),
            'als_score': np.concatenate(scores)
        })
        df_pred = (
            df_pred
            .with_columns(
                pl.col('als_score')
                .rank('ordinal', descending=True).over('cookie')
                .alias('als_rank')
            )
        )
        return df_pred

In [6]:
class FeatureGenerator:
    """
    Generate cookie- and node-level features from clickstream.
    """
    
    def __init__(self):
        self.cookie_features = None
        self.node_features = None
        self.category_features = None
        self.cookie_category_features = None
    
    def fit(
        self, 
        df_clickstream: pl.DataFrame, 
        df_cat: pl.DataFrame, 
        mode: str
    ):
        self.df = (
            df_clickstream
            .join(df_cat.select(['item', 'location', 'category']), on='item', how='left')
            .fill_null(1)
        )
        self.cookie_features = (
            self.df
            .select('cookie').unique('cookie').sort('cookie')
            .join(self.get_obj_events_contacts(obj='cookie'), on='cookie', how='left')
            .join(self.get_obj_items(obj='cookie'),           on='cookie', how='left')
            .join(self.get_obj_location_top(obj='cookie'),    on='cookie', how='left')
            .join(self.get_obj_location_count(obj='cookie'),  on='cookie', how='left')
            .join(self.get_cookie_recency(),                  on='cookie', how='left')
        )
        self.node_features = (
            self.df
            .select('node').unique('node').sort('node')
            .join(self.get_obj_events_contacts(obj='node'), on='node', how='left')
            .join(self.get_obj_items(obj='node'),           on='node', how='left')
            .join(self.get_obj_location_count(obj='node'),  on='node', how='left')
        )
        self.category_features = (
            self.df
            .select('category').unique('category').sort('category')
            .join(self.get_obj_events_contacts(obj='category'), on='category', how='left')
            .join(self.get_obj_items(obj='category'),           on='category', how='left')
            .join(self.get_obj_location_count(obj='category'),  on='category', how='left')
            .join(self.get_category_nodes(),                    on='category', how='left')
        )
        self.cookie_category_features = self.get_cookie_category_features()
        
        del self.df
        gc.collect()
        return self
        
    
    def transform(self, df: pl.DataFrame) -> pl.DataFrame:
        return (
            df
            .join(self.cookie_features,   on='cookie',   how='left')
            .join(self.node_features,     on='node',     how='left')
            .join(self.category_features, on='category', how='left')
            .join(self.cookie_category_features, on=['cookie', 'category'], how='left')
        )
    
    def get_obj_events_contacts(self, obj: str) -> pl.DataFrame:
        df_feats = self.df.select(obj).unique(obj)
        max_date = self.df["event_date"].max()
        periods = {"all": None, "7d": 7, "14d": 14}
        
        for suffix, days in periods.items():
            cur_df = self.df if days is None else (
                self.df
                .filter(pl.col("event_date") >= max_date - timedelta(days=days))
            )
            cur_df = (
                cur_df
                .select([obj, 'is_contact'])
                .group_by(obj)
                .agg([
                    pl.col('is_contact').sum().alias(f"{obj}_contacts_{suffix}"),
                    pl.count().alias(f'{obj}_events_{suffix}')
                ])
                .with_columns(
                    (pl.col(f'{obj}_contacts_{suffix}') / pl.col(f'{obj}_events_{suffix}'))
                    .alias(f'{obj}_contact_rate_{suffix}')
                )
                .drop(f'{obj}_events_{suffix}')
                .with_columns([
                    pl.col(f'{obj}_contacts_{suffix}').shrink_dtype(),
                    pl.col(f'{obj}_contact_rate_{suffix}').shrink_dtype()
                ])
            )
            df_feats = df_feats.join(cur_df, on=obj, how='left')
            
        return df_feats.fill_null(0)
    
    def get_cookie_category_features(self) -> pl.DataFrame:
        """
        Contacts and contact rate per cookie-category pair.
        """
        return (
            self.df
            .select(['cookie', 'category', 'is_contact'])
            .group_by(['cookie', 'category'])
            .agg([
                pl.col('is_contact').sum().alias('cookie_category_contacts'),
                pl.count().alias('cookie_category_events')
            ])
            .with_columns(
                (pl.col('cookie_category_contacts') / pl.col('cookie_category_events'))
                .alias('cookie_category_contact_rate')
            )
            .fill_null(0)
            .with_columns([
                pl.col('cookie_category_events').shrink_dtype(),
                pl.col('cookie_category_contacts').shrink_dtype(),
                pl.col('cookie_category_contact_rate').shrink_dtype()
            ])
        )
    
    def get_obj_items(self, obj: str) -> pl.DataFrame:
        """
        Кол-во уникальных items у obj (cookie, node или category)
        """
        return (
            self.df
            .select([obj, 'item'])
            .group_by(obj)
            .agg(pl.col('item').n_unique().alias(f'{obj}_items'))
            .with_columns(pl.col(f'{obj}_items').shrink_dtype())
        )
    
    def get_obj_location_top(self, obj: str) -> pl.DataFrame:
        """
        Самая популярная локация у obj (cookie, node или category)
        """
        return (
            self.df
            .select([obj, 'location'])
            .group_by([obj, 'location'])
            .agg(pl.count().alias("cnt"))
            .sort([obj, "cnt"], descending=[False, True])
            .group_by(obj)
            .agg(pl.first('location').alias(f"{obj}_location_top"))
            .with_columns(pl.col(f'{obj}_location_top').shrink_dtype())
        )
    
    def get_obj_location_count(self, obj: str) -> pl.DataFrame:
        """
        Кол-во локаций у obj (cookie, node или category)
        """
        return (
            self.df
            .select([obj, 'location'])
            .group_by(obj)
            .agg(pl.col('location').n_unique().alias(f"{obj}_location_count"))
            .select([obj, f"{obj}_location_count"])
            .with_columns(pl.col(f"{obj}_location_count").shrink_dtype())
        )
    
    def get_cookie_recency(self) -> pl.DataFrame:
        """
        Кол-во дней с последнего event у cookie
        """
        max_date = self.df['event_date'].max()
        
        return (
            self.df
            .select(['cookie', 'event_date'])
            .group_by('cookie')
            .agg(pl.col('event_date').max().alias('last_event'))
            .with_columns(
                ((max_date - pl.col('last_event')).dt.total_seconds())
                .alias('cookie_recency_event')
            )
            .with_columns(
                (pl.col("cookie_recency_event") / pl.col("cookie_recency_event").max())
                .alias("cookie_recency_event")
            )
            .select(['cookie','cookie_recency_event'])
            .with_columns(pl.col("cookie_recency_event").shrink_dtype())
            
        )
    
    def get_category_nodes(self) -> pl.DataFrame:
        """
        Кол-во nodes внутри category
        """
        return (
            self.df
            .select(['category', 'node'])
            .group_by('category')
            .agg(pl.col('node').n_unique().alias('category_nodes'))
            .with_columns(pl.col("category_nodes").shrink_dtype())
        )


In [7]:
class RankerCatBoost:
    def __init__(self, params: dict[str, float]):
        self.params = {
            key[3:]: value 
            for key, value in params.items() 
            if key.startswith('cb_')
        }
        self.params_without_val = self.params | dict(
            task_type = DEVICE,
            eval_metric = 'RecallAt:top=40',
            nan_mode = "Min",
            metric_period = 10,
            random_seed = SEED, 
            thread_count = -1
        )
        self.params_with_val = self.params_without_val | dict(
            early_stopping_rounds = 100, 
            use_best_model = True
        )

        self.features = None
        self.best_iteration = None
        self.model = None

    
    def fit(self, df_train: pl.DataFrame, df_val: pl.DataFrame):
        self.features = [
            col 
            for col in df_train.columns 
            if col not in ('cookie', 'node', 'label')
        ]

        train_pool = Pool(
            data     = df_train[self.features].to_pandas(),
            label    = df_train['label'].to_pandas(),
            group_id = df_train['cookie'].to_pandas(),
        )
        val_pool = Pool(
            data     = df_val[self.features].to_pandas(),
            label    = df_val['label'].to_pandas(),
            group_id = df_val['cookie'].to_pandas(),
        )

        self.model = CatBoostRanker(**self.params_with_val)
        self.model.fit(
            train_pool,
            eval_set=val_pool,
            verbose=100
        )
        self.params_without_val['iterations'] = self.model.get_best_iteration()
        
        # Feature Importance
        # df_fi = self.model.get_feature_importance(data=train_pool, type='LossFunctionChange', prettified=True)
        # df_fi['Importances'] = df_fi['Importances'] * 100
        # df_fi.to_csv('feature_importance.csv', index=False)
        # mlflow.log_artifact('/kaggle/working/feature_importance.csv')
        # print(df_fi)
        
        return self

    
    def refit_full(self, df_full: pl.DataFrame):
        full_pool = Pool(
            data     = df_full[self.features].to_pandas(),
            label    = df_full['label'].to_pandas(),
            group_id = df_full['cookie'].to_pandas(),
        )
        
        self.model = CatBoostRanker(**self.params_without_val)
        self.model.fit(
            full_pool,
            verbose=100
        )
        return self

    
    def predict(self, df_cand: pl.DataFrame) -> pl.DataFrame:
        """
        Score df_cand and return top-40 per cookie.
        """
        result = []
        chunk_size = 100_000
        for i in tqdm(range(0, df_cand.height, chunk_size), desc='Scoring Catboost'):
            chunk = df_cand[i:i + chunk_size]
            scores = self.model.predict(chunk[self.features].to_pandas())
            chunk = (
                chunk
                .select(['cookie', 'node'])
                .with_columns(pl.Series(scores).alias('ranker_score'))
                )
            result.append(chunk)
            
            del chunk, scores
            gc.collect()
        
        df_pred = pl.concat(result)
        return (
            df_pred
            .sort(['cookie', 'ranker_score'], descending=[False, True])
            .group_by('cookie')
            .head(40)
        )


In [8]:
def get_projection_scores(
    df_cands: pl.DataFrame, 
    cookie_projection: pl.DataFrame, 
    node_projection: pl.DataFrame,
    mode: str
) -> pl.DataFrame:
    """
    Add projection scores to candidates.
    """

    bufs = []
    chunk_size = 100_000
    for start in tqdm(range(0, df_cands.height, chunk_size), mininterval=60):
        block = df_cands[start:start+chunk_size]
        df_block = (
            block
            .join(node_projection,   on='node', how='left')
            .join(cookie_projection, on='cookie', how='left')
            .with_columns(
                pl.struct('cookie_projection', 'node_projection')
                .map_elements(lambda cols: np.dot(cols['cookie_projection'], cols['node_projection']))
                .alias('cosine_similarity')
            )
            .drop(['cookie_projection', 'node_projection'])
        )
        bufs.append(df_block)

    df_projection_scores = pl.concat(bufs)
    return df_projection_scores


def get_cookie_node_projections(
    df_clickstream: pl.DataFrame,
    df_to_pred: pl.DataFrame,
    mode: str,
) -> pl.DataFrame:
    """
    Load or compute cookie- and node-level projections.
    """
    node_projection = (
        pl.read_parquet(PROJECTION_NODE_PATH)
        .filter(pl.col('node').is_in(df_clickstream['node'].unique()))
    )
    
    cookie_projection_path = FILES[mode]['projection_cookie']
    if cookie_projection_path is not None:
        cookie_projection = (
            pl.read_parquet(cookie_projection_path)
            .filter(pl.col('cookie').is_in(df_to_pred['cookie'].unique()))
        )
    
    else:
        cookie_projection = (
            df_clickstream
            .select(['cookie', 'node'])
            .filter(pl.col('cookie').is_in(df_to_pred['cookie'].unique()))
            .unique()
            .join(node_projection, on='node', how='left')
            .group_by('cookie')
            .agg(
                pl.concat_list(
                    pl.col("node_projection")
                    .list.slice(n, 1)
                    .list.first()
                    .mean()
                    for n in range(0, 64)
                ).alias('cookie_projection')
            )
        )
        cookie_projection.write_parquet(f'projection_cookie_{mode}_v{VERSION}.parquet')
    
    return cookie_projection, node_projection

In [9]:
def get_cands(
    df_clickstream: pl.DataFrame, 
    df_to_pred: pl.DataFrame,
    df_cat: pl.DataFrame,
    mode: str = 'VAL'
) -> pl.DataFrame:
    """
    Generate combined candidate set from ALS, popularity, and text projection.
    """
    cookie_projection, node_projection = get_cookie_node_projections(df_clickstream, df_to_pred, mode)
    
    df_als        = get_cands_als(df_clickstream, df_to_pred, mode)
    df_lightfm    = get_cands_lightfm(df_clickstream, df_to_pred, mode)
    df_popular    = get_cands_popular(df_clickstream, df_to_pred, mode)
    df_projection = get_cands_projection(df_clickstream, df_to_pred, mode, cookie_projection, node_projection)

    df_cands = (
        df_als
        .join(df_lightfm,    on=['cookie', 'node'], how='full', coalesce=True)
        .join(df_popular,    on=['cookie', 'node'], how='full', coalesce=True)
        .join(df_projection, on=['cookie', 'node'], how='full', coalesce=True)
        .sort(['cookie', 'node'], descending=[False, False])
    )

    df_cands = filter_cands_not_seen(df_cands, df_clickstream)
    df_cands = get_projection_scores(df_cands, cookie_projection, node_projection, mode)
    df_cands = get_category_to_node(df_cands, df_cat)
    return df_cands
    

def get_cands_als(
    df_clickstream: pl.DataFrame, 
    df_to_pred: pl.DataFrame, 
    mode: str
) -> pl.DataFrame:
    """
    Load or compute ALS-based candidates.
    """
    N_cands = PARAMS['als_N_cand']
    als_path = FILES[mode]['cands_als']
    
    if als_path is not None:
        return (
            pl.read_parquet(als_path)
            .group_by('cookie')
            .head(N_cands)
        )
    
    als = ALSRecommender(PARAMS)
    als.fit(df_clickstream)
    df_cands_als = als.recommend(df_to_pred)
    df_cands_als.write_parquet(f'cands_als_{mode}_v{VERSION}_n{N_cands}.parquet')
    return df_cands_als


def get_cands_lightfm(
    df_clickstream: pl.DataFrame,
    df_to_pred: pl.DataFrame,
    mode: str
) -> pl.DataFrame:
    """
    Load or compute LightFM-based candidates.
    """
    N_cands = PARAMS['lightfm_N_cand']
    lightfm_path = FILES[mode]['cands_lightfm']
    
    if lightfm_path is not None:
        return (
            pl.read_parquet(lightfm_path)
            .group_by('cookie')
            .head(N_cands)
        )
    
    df_clickstream = df_clickstream.filter(pl.col('is_contact') == 1)
    df_to_pred = (
        df_to_pred
        .filter(pl.col('cookie').is_in(df_clickstream['cookie'].unique()))
        ['cookie'].unique()
    )
    
    cookie_features = (
        pl.read_parquet(FILES[mode]['cookie_features'])
        .filter(pl.col('cookie').is_in(df_clickstream['cookie'].unique()))
    )
    node_features = (
        pl.read_parquet(FILES[mode]['node_features'])
        .filter(pl.col('node').is_in(df_clickstream['node'].unique()))
    )
    
    dataset = Dataset()
    dataset.fit(
        users=df_clickstream['cookie'].to_list(),
        items=df_clickstream['node'].to_list(),
        user_features=cookie_features.columns,
        item_features=node_features.columns
    )
    
    (interactions, weights) = dataset.build_interactions(
        [(row['cookie'], row['node']) for row in df_clickstream.rows(named=True)]
    )
    user_features = dataset.build_user_features([
        (row['cookie'], {k: v for k, v in row.items() if k != 'cookie'})
        for row in cookie_features.rows(named=True)
    ])
    item_features = dataset.build_item_features([
        (row['node'], {k: v for k, v in row.items() if k != 'node'})
        for row in node_features.rows(named=True)
    ])
    
    model = LightFM(
        no_components=PARAMS['lightfm_factors'],
        loss=PARAMS['lightfm_loss'], 
        learning_rate=PARAMS['lightfm_lr'],
        random_state=SEED
    )
    model.fit(
        interactions,
        sample_weight=weights,
        user_features=user_features,
        item_features=item_features,
        epochs=PARAMS['lightfm_epochs'],
        num_threads=4
    )
    
    # Recommend
    user_ids_map = dataset.mapping()[0]
    item_ids_map = dataset.mapping()[2]
    inv_item_map = {v: k for k, v in item_ids_map.items()}

    interactions_csr = interactions.tocsr()
    user_seen = [interactions_csr[i].indices for i in range(interactions.shape[0])]
    
    rows = []
    for u in tqdm(df_to_pred, mininterval=60, desc='LightFM recommend'):
        u_id = user_ids_map[u]
        
        scores = model.predict(
            user_ids=u_id,
            item_ids=np.arange(len(item_ids_map)),
            user_features=user_features,
            item_features=item_features,
            num_threads=8
        )

        # masking seen items by cookie
        scores[user_seen[u_id]] = -np.inf
        
        top_items = np.argsort(-scores)[:PARAMS['lightfm_N_cand']]
        for rank, i_id in enumerate(top_items):
            rows.append({
                'cookie':u, 
                'node':inv_item_map[i_id], 
                'lightfm_score':float(scores[i_id]),
                'lightfm_rank': rank + 1
            })

    df_cands_lightfm = pl.DataFrame(rows)
    df_cands_lightfm.write_parquet(f'cands_lightfm_{mode}_v{VERSION}_n{N_cands}.parquet')
    return df_cands_lightfm
    

def get_cands_popular(
    df_clickstream: pl.DataFrame,
    df_to_pred: pl.DataFrame,
    mode: str
) -> pl.DataFrame:
    """
    Load or compute popularity-based candidates.
    """
    N_cands = PARAMS['pop_N_cand']
    pop_path = FILES[mode]['cands_pop']
    
    if pop_path is not None:
        df_popular_nodes = (
            pl.read_parquet(pop_path)
            .head(N_cands)
        )

    else:
        df = (
            df_clickstream
            .filter(pl.col('is_contact') == 1)
            .select(['node', 'cookie', 'event_date'])
        )
        max_date = df_clickstream['event_date'].max()
        df_popular_nodes = (
            df
            .filter(pl.col('event_date') >= max_date - pl.duration(days=14))
            .group_by('node')
            .agg(pl.count('cookie').alias('popular_score'))
            .sort('popular_score', descending=True)
            .head(N_cands)
            .select(['node']) #, 'popular_score'])
        )
        df_popular_nodes.write_parquet(f'cands_pop_{mode}_v{VERSION}_n{N_cands}.parquet')
    
    df_cands_popular = (
        df_to_pred.select('cookie').unique()
        .join(df_popular_nodes, how='cross')
    )
    return df_cands_popular


def get_cands_projection(
    df_clickstream: pl.DataFrame, 
    df_to_pred: pl.DataFrame, 
    mode: str,
    cookie_projection: pl.DataFrame,
    node_projection: pl.DataFrame
) -> pl.DataFrame:
    """
    Load or compute text-projection-based candidates using FAISS.
    """
    N_cands = PARAMS['proj_N_cand']
    proj_path = FILES[mode]['cands_proj']
    
    if proj_path is not None:
        return (
            pl.read_parquet(proj_path)
            # .sort(['cookie', 'text_score'], descending=[False, True])
            .group_by('cookie')
            .head(N_cands)
        )
    
    df_users = (
        df_to_pred
        .select('cookie')
        .unique()
        .sort('cookie')    
        .join(cookie_projection, on='cookie', how='left')
    )
    

    nodes = node_projection['node'].to_numpy()
    X = np.vstack(node_projection['node_projection'].to_list()).astype('float32')
    # faiss.normalize_L2(X)
    
    index = faiss.IndexFlatIP(X.shape[1])
    index.add(X)

    records = []
    for row in tqdm(df_users.rows(named=True), mininterval=60):
        u_id = row['cookie']
        u_vec = np.array(row['cookie_projection'], dtype='float32').reshape(1, -1)
        # faiss.normalize_L2(u_vec)
    
        D, I = index.search(u_vec, N_cands)
        for dist, idx in zip(D[0], I[0]):
            records.append({
                'cookie': u_id, 
                'node': int(nodes[idx]), 
                # 'text_score': float(dist)
            })

    df_cands_projection = pl.DataFrame(records)
    df_cands_projection.write_parquet(f'cands_proj_{mode}_v{VERSION}_n{N_cands}.parquet')
    return df_cands_projection


def filter_cands_not_seen(
    df_cands: pl.DataFrame,
    df_clickstream: pl.DataFrame
) -> pl.DataFrame:
    """
    Remove (cookie, node) pairs that appear in historical clickstream (train dataset).
    """
    return df_cands.join(df_clickstream, on=['cookie', 'node'], how='anti')


def get_category_to_node(df_cands: pl.DataFrame, df_cat: pl.DataFrame) -> pl.DataFrame:
    """
    Transform candidates: (cookie, node) -> (cookie, node, category)
    """
    df_node_to_category = (
        df_cat
        .select(['node', 'category'])
        .group_by('node')
        .agg(pl.col('category').first())
    )
    return (
        df_cands
        .join(df_node_to_category, on='node', how='left')
    )


## Pipeline

In [10]:
def do_val_pipeline():
    df_test, df_clickstream, df_cat = get_data()
    
    df_train, df_val = split_train_val(df_clickstream)
    df_cands = get_cands(df_train, df_val, df_cat, mode='VAL')
    df_cands = get_labels(df_val, df_cands)

    recall_cands = recall_candidates(df_val, df_cands)
    mlflow.log_metric('Recall_cands', recall_cands)
    print(f"Candidate Recall = {recall_cands:.5f}")
    
    neg_ration_cands = int(1 / df_cands['label'].mean())
    mlflow.log_metric('Neg_ratio_cands', neg_ration_cands)
    print(f"Cands negative ratio = {neg_ration_cands}")

    feature_gen = FeatureGenerator()
    feature_gen.fit(df_train, df_cat, mode='VAL')
    df_cands = feature_gen.transform(df_cands)
    print(f'df_cands size = {df_cands.estimated_size() / 1024**3:.1f} ГБ')
    
    del feature_gen, df_test, df_clickstream, df_cat, df_train
    gc.collect()
    
    df_cands_train, df_cands_val = split_cands_train_val(df_cands)
    ranker = RankerCatBoost(PARAMS)
    ranker.fit(df_cands_train, df_cands_val)
    mlflow.log_param('cb_best_iteration', ranker.model.get_best_iteration())
    del df_cands_train, df_cands_val
    gc.collect()
    
    df_predict = ranker.predict(df_cands)
    
    recall_40 = recall_at(df_val, df_predict, k=40)
    mlflow.log_metric('Recall_40', recall_40)
    print(f'Validation Recall_40 = {recall_40:.5f}')

    return ranker, df_cands


def do_test_pipeline():
    df_test, df_clickstream, df_cat = get_data()
    
    df_cands_test = get_cands(df_clickstream, df_test, df_cat, mode='TEST')

    feature_gen = FeatureGenerator()
    feature_gen.fit(df_clickstream, df_cat, mode='TEST')
    df_cands_test = feature_gen.transform(df_cands_test)
    print(f'df_cands_test size = {df_cands_test.estimated_size() / 1024**3:.1f} ГБ')
    
    return df_cands_test

In [11]:
with mlflow.start_run(run_name=f"als_lightfm_pop_proj + cb ({VERSION})"):
    mlflow.log_params(PARAMS)
    
    ranker, df_cands = do_val_pipeline()
    
    if PARAMS['do_test_pred']:
        ranker.refit_full(df_cands)
        del df_cands
        gc.collect()
        
        df_cands_test = do_test_pipeline()
        
        df_predict = ranker.predict(df_cands_test)
        df_predict.select(['cookie','node']).write_csv(f'submission_{VERSION}.csv')
        

100%|██████████| 395/395 [09:39<00:00,  1.47s/it]


Candidate Recall = 0.56594
Cands negative ratio = 486
df_cands size = 5.3 ГБ


Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric RecallAt:top=40 is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.4391280	test: 0.4390739	best: 0.4390739 (0)	total: 15.6s	remaining: 4h 19m 46s
100:	learn: 0.4971181	test: 0.4929553	best: 0.4931752 (91)	total: 2m 10s	remaining: 19m 21s
200:	learn: 0.5014089	test: 0.4942858	best: 0.4945666 (170)	total: 4m 6s	remaining: 16m 18s
300:	learn: 0.5044869	test: 0.4969884	best: 0.4969884 (300)	total: 6m 1s	remaining: 13m 59s
400:	learn: 0.5080173	test: 0.4979924	best: 0.4982104 (399)	total: 7m 56s	remaining: 11m 52s
500:	learn: 0.5095164	test: 0.4983744	best: 0.4989327 (489)	total: 9m 51s	remaining: 9m 49s
600:	learn: 0.5123492	test: 0.5000840	best: 0.5000900 (597)	total: 11m 47s	remaining: 7m 49s
700:	learn: 0.5138529	test: 0.5013919	best: 0.5019050 (680)	total: 13m 42s	remaining: 5m 50s
800:	learn: 0.5152181	test: 0.5022351	best: 0.5024109 (788)	total: 15m 38s	remaining: 3m 53s
900:	learn: 0.5169595	test: 0.5024450	best: 0.5028435 (806)	total: 17m 33s	remaining: 1m 55s
bestTest = 0.5028435388
bestIteration = 806
Shrink model to first 807 iterat

Scoring Catboost: 100%|██████████| 395/395 [02:14<00:00,  2.93it/s]


...

0:	learn: 0.4269197	total: 2.14s	remaining: 28m 44s
100:	learn: 0.4978610	total: 2m 9s	remaining: 15m 5s
200:	learn: 0.5017535	total: 4m 18s	remaining: 12m 57s
300:	learn: 0.5045618	total: 6m 26s	remaining: 10m 48s
400:	learn: 0.5068810	total: 8m 35s	remaining: 8m 40s
500:	learn: 0.5097261	total: 10m 43s	remaining: 6m 31s
600:	learn: 0.5117472	total: 12m 51s	remaining: 4m 23s
700:	learn: 0.5132657	total: 14m 59s	remaining: 2m 14s
800:	learn: 0.5145764	total: 17m 7s	remaining: 6.41s
805:	learn: 0.5146133	total: 17m 13s	remaining: 0us


100%|██████████| 644/644 [15:57<00:00,  1.49s/it]


df_cands_test size = 8.7 ГБ


Scoring Catboost: 100%|██████████| 644/644 [03:59<00:00,  2.69it/s]


🏃 View run als_lightfm_pop_proj + cb (50) at: http://51.250.35.156:5000/#/experiments/27/runs/3e2829e0ea9b43b8b9b57e304c83e4c2
🧪 View experiment at: http://51.250.35.156:5000/#/experiments/27
